# 01 · RAAMove — build the pool

*Rhetorical moves in research-article abstracts (8 classes)*

### Where this sits

```
▶ 01 build the pool  →  02 sample  →  03 annotate  →  04 prompt  →  05 report
```

You run **01 once per group**, for your own track only. It ends by writing `data/pools/<track>_pool.json` — the file notebook 02 opens.

---

**What it is.** 400 RA abstracts, annotated sentence by sentence with one of eight rhetorical moves (Background, Gap, Purpose, Method, Result, Conclusion, Contribution, Implication). Reported annotator agreement: κ = 0.785.

**Difficulty of the labeling judgment:** ★★☆ — moderate. Moves are functional categories, so neighbouring sentences can be genuinely hard to separate.

**Licence:** CC BY 4.0  
**Cite:** Liu, J. et al. (2024), *LREC-COLING*. github.com/ljk1228/RAAMove

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

> The reshaping code below is read straight out of `scripts/reshape.py` — it is the same code `scripts/prep_datasets.py` runs, not a copy of it. What is *missing* from it is missing on purpose: the ✏️ cells are the decisions, and they are yours. (Generated by `scripts/_generate_pool_notebooks.py`; edit that or `reshape.py`, never the `.ipynb`.)

## Step 1 — Download the raw data

In [ ]:
!git clone --depth 1 https://github.com/ljk1228/RAAMove

## Step 2 — Look at the raw format

This one is **JSON**, split into two files by discipline (`Intelligence.json`, `Engineering.json`). Each record has a `text` and a three-letter move code in `labels`.

**Read the codes the cell below prints** — you are about to name every one.

In [ ]:
RAW_DIR = "RAAMove"

import json
from collections import Counter

data = json.loads(open(RAW_DIR + "/Intelligence.json", encoding="utf-8").read())
print("records:", len(data))
print("codes:", Counter(record["labels"] for record in data))
data[:3]

## Step 3 — Reshape into the canonical schema

Two decisions:

1. ✏️ **What each code is called.** `BAC` → `Background`. The expansion is not cosmetic: it is the wording your prompt will use, your annotators will read on the sheet, and your confusion matrix will be labelled with. `Gap` and `Establishing a niche` describe the same category and will not get you the same predictions.
2. **Pool the two disciplines** — the code below reads both files into one set, treating a move as a rhetorical function rather than a discipline-specific one. That *is* an assumption. It is in the code rather than in a ✏️ cell only because unpicking it makes a better extension than a starting point: comparing Intelligence against Engineering separately would be a real finding.

In [ ]:
# ✏️ Step 3a · Name the moves ────────────────────────────────────
# Goal      : map each three-letter code to the move name your prompt will use.
# Shape     : RAAMOVE_LABELS = {"BAC": "Background", ...}
#             one entry per code you saw in step 2
# Produce   : RAAMOVE_LABELS (a dict)      ← later cells use this name
# Careful   : a code you leave out is NOT dropped — it is kept as the raw
#             code, so it shows up as a stray label like "CTN" in step 4.
#             That is your check that you got them all.
# Note      : eight classes is a lot. If you plan to merge any (Result +
#             Conclusion, say), decide it HERE and say so in PLAN.md — not
#             after you have seen the model do badly on them.

# ✏️ your code here


The function below reads the `RAAMOVE_LABELS` you just defined.

In [ ]:
import json
from pathlib import Path

def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1
    for item in items:
        new_item = dict(item)
        new_item["id"] = next_id
        renumbered.append(new_item)
        next_id = next_id + 1
    return renumbered

def reshape_raamove(raamove_dir):
    """Read RAAMove's per-domain JSON files and expand the 3-letter move codes.

    The corpus ships two domains (Intelligence, Engineering) as separate files. We pool
    them, because a move is meant to be a rhetorical function rather than a
    discipline-specific one - but that IS an assumption, and comparing the two domains
    separately would be a perfectly good extension.
    """
    source_dir = Path(raamove_dir)
    rows = []
    for filename in ("Intelligence.json", "Engineering.json"):
        path = source_dir / filename
        if not path.exists():
            continue
        data = json.loads(path.read_text(encoding="utf-8"))
        for record in data:
            code = record["labels"]
            if code in RAAMOVE_LABELS:
                label = RAAMOVE_LABELS[code]
            else:
                label = code            # an unexpected code: keep it and let validate() complain
            rows.append({"id": 0, "text": record["text"].strip(), "label": label})
    return reid(rows)

In [ ]:
rows = reshape_raamove(RAW_DIR)

## Step 4 — Check the label balance

Very imbalanced: `Method` is the biggest class by far, and `Implication` has only a couple of dozen sentences.

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
rows[:3]        # peek at the first three reshaped items

In [ ]:
# ✏️ Step 4b · React to the balance ──────────────────────────────
# Goal      : decide what the counts you just printed mean for your study.
# Shape     : MIN_PER_CLASS = <the size of your SMALLEST class>
#             that is the ceiling on N_PER_CLASS in config.py — a balanced
#             sample cannot draw more from a class than the class has
# Produce   : MIN_PER_CLASS (an int)      ← later cells use this name
# Note      : with eight classes and a rare tail, the smallest class decides everything — 7 per class is about the most this pool supports.
# Note      : if the rarest class is tiny, say so in PLAN.md. Merging it
#             away or living with fewer items are both defensible;
#             not noticing is not.

# ✏️ your code here


## Step 5 — Save it

In [ ]:
# Save the pool. Two places you might want it:
#   * this repo, if you cloned it:  "../data/pools/raamove_pool.json"
#   * your Google Drive, so it survives the Colab runtime resetting
import json, pathlib

OUT_FILE = "../data/pools/raamove_pool.json"

# In Colab WITHOUT the repo, uncomment these two to write straight to Drive:
# from google.colab import drive; drive.mount("/content/drive")
# OUT_FILE = "/content/drive/MyDrive/raamove_pool.json"

pathlib.Path(OUT_FILE).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", OUT_FILE)

## What you just built, and what happens to it

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is **not** your gold set, and its labels are **not** your labels: they are the original corpus authors' judgment, and you have not yet agreed with them about anything.

What those labels are for is narrow, and worth being precise about:

1. **Stratifying the draw** in notebook 02 — you cannot sample evenly across classes without knowing what the classes are.
2. **A comparison** in notebook 03 — once you have annotated blind and adjudicated, `compare_to_published` shows you every item where your group landed somewhere different. That gap is evidence, and one of the more interesting things you can put in a report.

They are never the answer key you score the model against. That file does not exist yet — you make it in notebook 03.

---

**Next:** set `TRACK = "raamove"` in `config.py`, then open `02_sample.ipynb`.